In [ ]:
%run initialize_environment.py


# Multi-Agent Event Planner

In [ ]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

llm = create_azure_llm()
tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [ ]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


In [ ]:
# Venue agent
venue_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [ ]:
# Playlist agent
playlist_agent = create_agent(
    model=llm,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


In [ ]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [ ]:
from langchain.agents import create_agent

coordinator = create_agent(
    model=create_azure_llm(),
    tools=[search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a event planner & coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


## Test


In [9]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from Lahore and I'd like a wedding in Gujranwala for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)
print(response["messages"][-1].content)

For your wedding in Gujranwala with 100 guests and a jazz music theme, here are some excellent venue options:

1. Adam Palace Gujranwala
   - Capacity: 100 to 300 guests
   - Facilities: Central AC, heating, prime location on GT Road
   - Known for beautiful decoration, comfortable AC hall, and great food service
   - Affordable rates (please contact for exact pricing)

2. Golf Club Marquee, Rahwali Cantt, Gujranwala
   - Suitable for weddings and events
   - Pricing: Approx. Rs. 1150 per head plus additional cooking and kitchen charges; beverages extra
   - Praised for luxury, elegance, and excellent seating plans

3. Al Karam Hotel, Gujranwala
   - Capacity: Up to 3000 guests (can easily accommodate 100 guests)
   - Elegant interiors, outdoor spaces, and stunning decor
   - One of the best marquees in Gujranwala with perfect service
   - Contact for booking and pricing

4. The Signature Club, Gujranwala
   - Suitable for weddings and events
   - Noted for exquisite decor and flawless

In [10]:
response = await coordinator.ainvoke(
    {
        "messages": response["messages"] + [
            HumanMessage(
                content="Please share the playlist (song names, durations, prices, total duration, and total cost)."
            )
        ]
    },
    config={"tags": ["WP"], "recursion_limit": 40},
)

print(response["messages"][-1].content)

Here is the jazz playlist curated for your wedding, including song names, durations, and prices:

1. Antôno Carlos Jobim - Duration: 185,338 ms (~3:05) - Price: $0.99
2. Antôno Carlos Jobim - Duration: 285,048 ms (~4:45) - Price: $0.99
3. Antôno Carlos Jobim - Duration: 137,273 ms (~2:17) - Price: $0.99
4. Antôno Carlos Jobim - Duration: 169,900 ms (~2:50) - Price: $0.99
5. Antôno Carlos Jobim - Duration: 251,977 ms (~4:12) - Price: $0.99
6. Antôno Carlos Jobim - Duration: 129,227 ms (~2:09) - Price: $0.99
7. Antôno Carlos Jobim - Duration: 253,178 ms (~4:13) - Price: $0.99
8. Antôno Carlos Jobim - Duration: 134,948 ms (~2:15) - Price: $0.99
9. Antôno Carlos Jobim - Duration: 219,663 ms (~3:40) - Price: $0.99
10. Antôno Carlos Jobim - Duration: 169,508 ms (~2:50) - Price: $0.99

There are also additional tracks by artists like Billy Cobham, Spyro Gyra, Miles Davis, Gene Krupa, Dennis Chambers, Gilberto Gil, Incognito, Aisha Duo, and Aaron Goldberg available.

Total duration of the abov